# 10.4 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/04-docker/solutions.ipynb)

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


In [2]:
docker_dir = repo_file("modules/10-deployment-mlops/04-docker/Dockerfile").parent
dockerfile = (docker_dir / "Dockerfile").read_text()
ignore = (docker_dir / "Dockerfile.dockerignore").read_text()
requirements = (docker_dir / "requirements-api.txt").read_text()

## Exercise 1

Docker reuses every step up to the first one whose inputs changed, then redoes that step and **everything after it**.

- **(a) Retrained model**: copy the model, `ENV`, `useradd`, `USER`, `EXPOSE`, `CMD`. Libraries are reused: seconds.
- **(b) Typo in `api.py`**: copy `api.py` and everything after. Same: seconds.
- **(c) FastAPI upgrade**: copy requirements, `pip install`, and everything after. The slow step reruns.

This is why the slow, rarely-changing step comes first and the often-changing files come last.

## Exercise 2

In [3]:
dockerfile_app = """FROM python:3.12-slim
WORKDIR /app
COPY modules/10-deployment-mlops/04-docker/requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt streamlit==1.66.1
COPY lib/churn_model.py lib/
COPY modules/10-deployment-mlops/03-streamlit/app.py .
COPY models/churn_bundle.joblib models/
ENV CHURN_MODEL_PATH=/app/models/churn_bundle.joblib
RUN useradd --create-home appuser
USER appuser
EXPOSE 8501
CMD ["python", "-m", "streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.headless=true"]
"""
print(dockerfile_app)

FROM python:3.12-slim
WORKDIR /app
COPY modules/10-deployment-mlops/04-docker/requirements-api.txt .
RUN pip install --no-cache-dir -r requirements-api.txt streamlit==1.66.1
COPY lib/churn_model.py lib/
COPY modules/10-deployment-mlops/03-streamlit/app.py .
COPY models/churn_bundle.joblib models/
ENV CHURN_MODEL_PATH=/app/models/churn_bundle.joblib
RUN useradd --create-home appuser
USER appuser
EXPOSE 8501
CMD ["python", "-m", "streamlit", "run", "app.py", "--server.address=0.0.0.0", "--server.headless=true"]



It needs `streamlit` (pinned like everything else). `--server.address=0.0.0.0` makes it listen on all network interfaces, not just inside the container; `--server.headless=true` stops it trying to open a browser. FastAPI and uvicorn are not needed for the app but do no harm; a tidy version would have its own requirements file. Remember to add the new files to an ignore file for this Dockerfile too (`Dockerfile.app.dockerignore`).

## Exercise 3

In [4]:
import shutil, subprocess
docker = shutil.which("docker")
running = docker is not None and subprocess.run([docker, "info"], capture_output=True).returncode == 0
print("docker installed:", docker is not None, "| engine running:", running)

import time, json, urllib.request
if running:
    docker_dir = repo_file("modules/10-deployment-mlops/04-docker/Dockerfile").parent
    subprocess.run([docker, "build", "-q", "-f", str(docker_dir / "Dockerfile"), "-t", "shopwave-churn", str(ROOT)], check=True)
    cid = subprocess.run([docker, "run", "-d", "--rm", "-p", "8000:8000", "shopwave-churn"],
                         capture_output=True, text=True).stdout.strip()
    time.sleep(5)
    body = json.dumps({"orders": 2, "spend": 310, "products": 14, "tenure_days": 150, "days_since_last": 95}).encode()
    request = urllib.request.Request("http://127.0.0.1:8000/predict", body, {"Content-Type": "application/json"})
    print(json.load(urllib.request.urlopen(request)))
    subprocess.run([docker, "stop", cid], capture_output=True)
else:
    print("Docker is not running here; run the commands from the lesson in a terminal.")

docker installed: True | engine running: False
Docker is not running here; run the commands from the lesson in a terminal.


Without a trained model the build stops at `COPY models/churn_bundle.joblib models/` with "not found". That is a feature: an image without a model would start and then crash on its first request. Train first (`train_churn.py`), then build.